# Pattern 5: Gateway + JWT Authentication (Cognito)

Use Cognito as the identity provider. The Gateway validates JWT tokens from Cognito
before allowing access to KB targets. This enables per-user identity without exposing the KB ID.

**What you get:** Per-user authentication, token-based access, no KB ID exposure.

**What Cognito controls:** *Who* the caller is — a request without a valid token never
reaches the KB.

**What it doesn't control:** *Which* documents come back (use metadata filters — Pattern 2 —
or a JWT-claim interceptor — Pattern 7 — for that).

## Prerequisites

- Run [Pattern 1](01-direct-sdk.ipynb) first — it creates the shared bucket, uploads the
  sample documents, and creates the KB execution role. (The setup cell here re-runs it
  idempotently, and additionally creates the **Gateway role**.)
- IAM permissions for Bedrock, AgentCore (`bedrock-agentcore-control`), Cognito, S3, and IAM.

## Architecture

```
Agent ──► JWT Token ──► Gateway (CUSTOM_JWT) ──► KB Target ──► Managed KB
  │                        │
  │                        └── Validates token via Cognito OIDC discovery
  └── Cognito User Pool (identity provider)
```

In [ ]:
import boto3
import time
import json
import util   # util.py in this folder — shared bucket + upload + roles

# --- Configuration ---
REGION = "us-west-2"
S3_BUCKET = "<existing-or-unique-name-for-your-kb-bucket->"
S3_PREFIX = "documents/"

session = boto3.Session()

# Reuse the SAME bucket + docs + KB execution role as Pattern 1 (idempotent),
# then create the Gateway role the AgentCore Gateway assumes to retrieve.
info = util.setup(
    bucket_name=S3_BUCKET,
    prefix=S3_PREFIX,
    metadata=util.SAMPLE_FILE_METADATA,
    region_name=REGION,
)
ROLE_ARN    = info["role_arn"]
S3_BUCKET   = info["bucket"]
S3_PREFIX   = info["prefix"]
GW_ROLE_ARN = util.create_gateway_role(region_name=REGION)

# Clients
cp = session.client("bedrock-agent", region_name=REGION)
dp = session.client("bedrock-agent-runtime", region_name=REGION)
ac = session.client("bedrock-agentcore-control", region_name=REGION)
cognito = session.client("cognito-idp", region_name=REGION)
S3_ACCOUNT = session.client("sts").get_caller_identity()["Account"]

print(f"boto3 {boto3.__version__}")
print(f"KB role:      {ROLE_ARN}")
print(f"Gateway role: {GW_ROLE_ARN}")


In [ ]:
# Step 1: Create Cognito User Pool + App Client + test user
pool = cognito.create_user_pool(
    PoolName=f"p5-jwt-{int(time.time())}",
    Policies={"PasswordPolicy": {
        "MinimumLength": 8, "RequireUppercase": False,
        "RequireLowercase": False, "RequireNumbers": False, "RequireSymbols": False
    }},
    Schema=[{
        "Name": "department", "AttributeDataType": "String",
        "Mutable": True, "Required": False,
        "StringAttributeConstraints": {"MinLength": "1", "MaxLength": "256"}
    }]
)
pool_id = pool["UserPool"]["Id"]
print(f"User Pool: {pool_id}")

client_resp = cognito.create_user_pool_client(
    UserPoolId=pool_id, ClientName="p5-client",
    ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
    GenerateSecret=False
)
client_id = client_resp["UserPoolClient"]["ClientId"]
print(f"App Client: {client_id}")

# Create test user with department attribute
cognito.admin_create_user(
    UserPoolId=pool_id, Username="testuser",
    UserAttributes=[
        {"Name": "custom:department", "Value": "engineering"},
        {"Name": "email", "Value": "test@example.com"}
    ],
    MessageAction="SUPPRESS"
)
cognito.admin_set_user_password(
    UserPoolId=pool_id, Username="testuser",
    Password="TestPass1", Permanent=True
)
print("Test user created: testuser (department=engineering)")

In [ ]:
# Step 2: Get JWT token via initiate_auth
auth = cognito.initiate_auth(
    ClientId=client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": "testuser", "PASSWORD": "TestPass1"}
)
id_token = auth["AuthenticationResult"]["IdToken"]
access_token = auth["AuthenticationResult"]["AccessToken"]
print(f"ID Token: {id_token[:50]}...")
print(f"Access Token: {access_token[:50]}...")

In [ ]:
# Step 3: Create Gateway with CUSTOM_JWT authorizer
discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{pool_id}/.well-known/openid-configuration"
print(f"Discovery URL: {discovery_url}")

# allowedAudience vs allowedClients: the authorizer verifies EVERY field you
# configure, but Cognito splits these claims across its two tokens — the ACCESS
# token carries `client_id` (and NO `aud`), while the ID token carries `aud`
# (and NO `client_id`). Requiring BOTH means neither token can ever pass → 403.
# We present the access token (Step 6), so we require only `allowedClients`.
gw_response = ac.create_gateway(
    name=f"p5-jwt-gw-{int(time.time())}",
    roleArn=GW_ROLE_ARN,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": discovery_url,
        "allowedClients": [client_id]
    }}
)

gw_id = gw_response["gatewayId"]
print(f"Gateway: {gw_id}")

# Wait for READY
gw_url = None
for _ in range(24):
    gw = ac.get_gateway(gatewayIdentifier=gw_id)
    if gw["status"] == "READY":
        gw_url = gw.get("gatewayUrl", "N/A")
        break
    time.sleep(5)
print(f"Status: {gw['status']}")
print(f"Gateway URL: {gw_url}")

In [ ]:
# Step 4: Create KB + Data Source + Ingest
response = cp.create_knowledge_base(
    name=f"p5-jwt-kb-{int(time.time())}",
    roleArn=ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "MANAGED",
        "managedKnowledgeBaseConfiguration": {}   # empty = managed default embedding
    }
)
kb_id = response["knowledgeBase"]["knowledgeBaseId"]
print(f"KB: {kb_id}")

for _ in range(30):
    if cp.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"] == "ACTIVE":
        break
    time.sleep(5)
print("KB ACTIVE")

response = cp.create_data_source(
    knowledgeBaseId=kb_id,
    name="s3-source",
    dataSourceConfiguration={
        "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
        "managedKnowledgeBaseConnectorConfiguration": {
            "connectorParameters": {
                "type": "S3",
                "version": "1",
                "connectionConfiguration": {
                    "bucketName": S3_BUCKET,
                    "bucketOwnerAccountId": S3_ACCOUNT
                },
                "filterConfiguration": {"inclusionPrefixes": [S3_PREFIX]},
                "deletionProtectionConfiguration": {"enableDeletionProtection": False}
            },
            "deletionProtectionConfiguration": {"deletionProtectionStatus": "DISABLED"}
        }
    },
    vectorIngestionConfiguration={
        "parsingConfiguration": {"parsingStrategy": "SMART_PARSING"}
    }
)
ds_id = response["dataSource"]["dataSourceId"]
print(f"DS: {ds_id}")

for _ in range(12):
    if cp.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]["status"] == "AVAILABLE":
        break
    time.sleep(5)
print("DS AVAILABLE")

response = cp.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
job_id = response["ingestionJob"]["ingestionJobId"]
for _ in range(40):
    job = cp.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
    )["ingestionJob"]
    if job["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)
print(f"Ingestion: {job['status']}")

In [ ]:
# Step 5: Create KB Target on the JWT Gateway
target_response = ac.create_gateway_target(
    gatewayIdentifier=gw_id,
    name="kb-retrieve",
    targetConfiguration={
        "mcp": {
            "connector": {
                "source": {"connectorId": "bedrock-knowledge-bases"},
                "configurations": [{
                    "name": "Retrieve",
                    # Tool description exposed to the agent over MCP — this is what
                    # the LLM reads to decide when to call this KB.
                    "description": (
                        "Search two corporate documents: (1) Octank Financial's 10-K annual "
                        "report — financial statements, asset/liability schedules, exhibits, and "
                        "investor disclosures; and (2) a U.S. tornado background & forecasting "
                        "report — where tornadoes form, annual frequency (~1,200/yr), and NOAA data."
                    ),
                    "parameterValues": {
                        "knowledgeBaseId": kb_id,
                        "retrievalConfiguration": {
                            "managedSearchConfiguration": {
                                "numberOfResults": 5
                            }
                        }
                    }
                }]
            }
        }
    },
    credentialProviderConfigurations=[
        {"credentialProviderType": "GATEWAY_IAM_ROLE"}
    ]
)

target_id = target_response["targetId"]
print(f"Target: {target_id}")

for _ in range(12):
    t = ac.get_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
    if t["status"] == "READY":
        break
    time.sleep(5)
print(f"Target status: {t['status']}")

In [ ]:
# Step 6: Retrieve THROUGH the JWT gateway using the official MCP client.
# A CUSTOM_JWT gateway is NOT SigV4-signed (that was the AWS_IAM gateway in
# Patterns 3/4). Here the caller presents a Cognito JWT in the Authorization
# header, and the gateway validates it via Cognito OIDC discovery BEFORE routing
# to the KB target. dp.retrieve() would bypass all of this by hitting Bedrock
# directly.
#
# We use the streamable-HTTP MCP client (`pip install mcp`) rather than a
# hand-rolled requests.post: it performs the required initialize() handshake and
# handles the SSE/streaming transport and notifications for us. The auth header
# goes on a pre-built httpx.AsyncClient passed via `http_client=` (the older
# `streamablehttp_client(..., headers=...)` form is deprecated in mcp >= 1.x).
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

# One MCP session: initialize -> tools/list -> tools/call (valid JWT).
async with httpx.AsyncClient(headers={"Authorization": f"Bearer {access_token}"}) as http_client:
    async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()                   # MCP handshake (required)

            # tools/list — the connector exposes each configured tool prefixed
            # with the target name, e.g. `kb-retrieve___Retrieve`. (The managed-KB
            # connector can also expose `AgenticRetrieveStream` — a multi-step,
            # streaming variant — when the target declares it; Step 5 declares
            # only "Retrieve", so that is the single tool an agent sees here.)
            listing = await session.list_tools()
            print(f"Gateway tools: {[t.name for t in listing.tools]}")
            tool_name = next(t.name for t in listing.tools if t.name.split("___")[-1] == "Retrieve")
            print(f"Using tool:    {tool_name}")

            # tools/call — retrieve through the gateway. The KB ID is bound to
            # the target, so we never send it.
            result = await session.call_tool(
                name=tool_name,
                arguments={"retrievalQuery": {"text": "What are Octank's key financial results?"}},
            )

            print(f"isError: {result.isError}")
            print("=== Retrieved via gateway (valid JWT) ===")
            print(result.content[0].text[:800] if result.content else result)

In [ ]:
# Step 6b: SAME MCP call as Step 6, but with NO bearer token — watch it get blocked.
# The gateway validates the JWT at the edge (Cognito OIDC), so an unauthenticated
# caller is rejected BEFORE the KB is ever touched. The MCP client surfaces that
# rejection as an HTTP 401/403, which we catch to print a clean result.
try:
    async with httpx.AsyncClient(headers={"Authorization": f"Bearer {access_token+"x"}"}) as http_client:       # <-- wrong Authorization header
        async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()               # never gets past the edge
                await session.list_tools()
    print("Unexpected: call was allowed without a token")
except* httpx.HTTPStatusError as eg:
    for e in eg.exceptions:
        code = e.response.status_code
        print(f"HTTP {code} → {'⛔ BLOCKED — no valid JWT' if code in (401, 403) else 'unexpected status'}")

## JWT Authentication Flow

1. Client authenticates with Cognito → receives JWTs (an **ID token** *and* an **access token**)
2. Client sends request to Gateway with `Authorization: Bearer <token>` (we send the **access token**)
3. Gateway fetches OIDC config from `discoveryUrl` → validates the signature, expiry, and — per the
   authorizer config in Step 3 — the **`client_id`** claim against `allowedClients`
4. If valid, the request proceeds to the KB target

## ID token vs. access token

Cognito issues two JWTs and they carry **different claims** — this matters, because the authorizer
verifies *every* field you configure, and a claim it's looking for must actually be in the token you send:

| Claim | ID token | Access token | Source | Validated by |
|---|:---:|:---:|---|---|
| `sub` | ✓ | ✓ | Cognito user ID | — |
| `client_id` | — | ✓ | App client ID | `allowedClients` |
| `aud` | ✓ | — | App client ID | `allowedAudience` |
| `email` | ✓ | — | User attribute | — |
| `custom:department` | ✓ | — | Custom attribute | — |
| `iss` | ✓ | ✓ | Pool issuer URL | — |

We send the **access token**, so the authorizer is configured with `allowedClients` (matches `client_id`).
Configuring `allowedAudience` as well would reject the access token, since it has no `aud` claim.

## Claim-Based Filtering

Custom claims like `custom:department` (present on the **ID token**) can be used downstream (via Lambda
interceptors) to inject metadata filters dynamically. For example, an interceptor could read the
`department` claim and add a metadata filter so users only see documents tagged for their department.


In [ ]:
# Cleanup — order: target → gateway → DS → KB → Cognito
ac.delete_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
time.sleep(3)
ac.delete_gateway(gatewayIdentifier=gw_id)
time.sleep(3)
cognito.delete_user_pool(UserPoolId=pool_id)
cp.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
cp.delete_knowledge_base(knowledgeBaseId=kb_id)
print(f"Deleted: Gateway {gw_id}, Cognito {pool_id}, KB {kb_id}")